In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [17]:
df = pd.read_csv('Scores.csv', sep=';', decimal=',')
print(df.head())

   Unnamed: 0 student_id  gender parental.level.of.education   subject  score
0           1     id_001  female                 high school      math   23.0
1           2     id_001  female                 high school  language   40.0
2           3     id_002  female                 high school      math   68.0
3           4     id_002  female                 high school  language   80.5
4           5     id_003    male                 high school      math   82.0


In [18]:
missing_data = df.isnull().sum()
print("\n--- Missing Values ---")
print(missing_data[missing_data > 0])


--- Missing Values ---
Series([], dtype: int64)


In [19]:
print(df['gender'].unique())
print(df['subject'].unique())

['female' 'male']
['math' 'language']


In [20]:
gender_avg = df.groupby(['gender', 'subject'])['score'].mean().unstack()
print("\n moyennes par genre et subject: ")
print(gender_avg)


 moyennes par genre et subject: 
subject   language       math
gender                       
female   73.004082  63.857143
male     64.931535  68.946058


In [21]:
df['score_cat'] = pd.cut(df['score'],
                     bins=[0, 50, 75, 100],
                     labels= [ 'Low (<50)', 'Medium (50-75)', 'High (>75)'],
                     include_lowest=True,
                     ordered=True)
print("\nScore categories distribution:")
print(df['score_cat'].value_counts())


Score categories distribution:
score_cat
Medium (50-75)    542
High (>75)        307
Low (<50)         123
Name: count, dtype: int64


In [22]:
score_order = ['Low (<50)', 'Medium (50-75)', 'High (>75)']
df['score_cat'] = pd.Categorical(
    df['score_cat'], 
    categories=score_order, 
    ordered=True)

In [23]:
crosstab = pd.crosstab(
    [df['subject'], df['score_cat']],
    df['gender'],
    margins=True,
    margins_name='Total'

)
print("\n Effectifs absolus:")
print(crosstab)


 Effectifs absolus:
gender                   female  male  Total
subject  score_cat                          
language Low (<50)           15    35     50
         Medium (50-75)     123   147    270
         High (>75)         107    59    166
math     Low (<50)           45    28     73
         Medium (50-75)     146   126    272
         High (>75)          54    87    141
Total                       490   482    972


In [24]:
crosstab_pr = pd.crosstab(
    [df['subject'], df['score_cat']],
    df['gender'],
    normalize= 'columns'

) * 100
print("\n Effectifs absolus par pourcentage:")
print(crosstab_pr.round(1).astype(str) + '%')


 Effectifs absolus par pourcentage:
gender                  female   male
subject  score_cat                   
language Low (<50)        3.1%   7.3%
         Medium (50-75)  25.1%  30.5%
         High (>75)      21.8%  12.2%
math     Low (<50)        9.2%   5.8%
         Medium (50-75)  29.8%  26.1%
         High (>75)      11.0%  18.0%


In [25]:
crosstab_pr1 = pd.crosstab(
    [df['subject'], df['score_cat']],
    df['gender'],
    normalize= 'index'

) * 100
print("\n Effectifs absolus par pourcentage:")
print(crosstab_pr1.round(1).astype(str) + '%')


 Effectifs absolus par pourcentage:
gender                  female   male
subject  score_cat                   
language Low (<50)       30.0%  70.0%
         Medium (50-75)  45.6%  54.4%
         High (>75)      64.5%  35.5%
math     Low (<50)       61.6%  38.4%
         Medium (50-75)  53.7%  46.3%
         High (>75)      38.3%  61.7%


In [ ]:
# VISUALISATION DES TESTS D'HYPOTHÈSE
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Tests Statistiques : Comparaison Hommes vs Femmes', fontsize=16, fontweight='bold')

subjects = ['language', 'math']
colors = {'female': '#FF69B4', 'male': '#4169E1'}

for idx, subject in enumerate(subjects):
    subset = df[df['subject'] == subject]
    female_data = subset[subset['gender'] == 'female']['score']
    male_data = subset[subset['gender'] == 'male']['score']
    
    # 1. BOXPLOT + MOYENNES
    ax1 = axes[idx, 0]
    sns.boxplot(data=subset, x='gender', y='score', ax=ax1, palette=colors, width=0.5)
    sns.stripplot(data=subset, x='gender', y='score', ax=ax1, color='black', alpha=0.3, size=4, jitter=True)
    
    # Ajouter moyennes
    female_mean = female_data.mean()
    male_mean = male_data.mean()
    ax1.scatter(0, female_mean, color='red', s=150, marker='★', label=f'Moyenne F: {female_mean:.1f}')
    ax1.scatter(1, male_mean, color='darkred', s=150, marker='★', label=f'Moyenne H: {male_mean:.1f}')
    ax1.legend(loc='upper right')
    ax1.set_title(f'{subject.title()} - Distribution des Scores', fontweight='bold', fontsize=12)
    ax1.set_ylabel('Score')
    ax1.set_xlabel('')
    
    # 2. BAR CHART - CATÉGORIES
    ax2 = axes[idx, 1]
    categories = ['Low (<50)', 'Medium (50-75)', 'High (>75)']
    crosstab_pct = pd.crosstab(subset['score_cat'], subset['gender'], normalize='columns') * 100
    
    x = np.arange(len(categories))
    width = 0.35
    
    female_pct = [crosstab_pct.loc[cat, 'female'] for cat in categories]
    male_pct = [crosstab_pct.loc[cat, 'male'] for cat in categories]
    
    bars1 = ax2.bar(x - width/2, female_pct, width, label='Femmes', color=colors['female'], edgecolor='black')
    bars2 = ax2.bar(x + width/2, male_pct, width, label='Hommes', color=colors['male'], edgecolor='black')
    
    ax2.set_ylabel('Pourcentage (%)')
    ax2.set_title(f'{subject.title()} - Répartition par Catégorie', fontweight='bold', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(categories, rotation=10, ha='right')
    ax2.legend()
    ax2.set_ylim(0, 100)
    
    # Valeurs sur les barres
    for bar in bars1:
        h = bar.get_height()
        ax2.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                    textcoords='offset points', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        h = bar.get_height()
        ax2.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                    textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 3. FOREST PLOT - EFFECT SIZE (COHEN'S D)
def calc_cohens_d(g1, g2):
    n1, n2 = len(g1), len(g2)
    v1, v2 = g1.var(ddof=1), g2.var(ddof=1)
    pooled = np.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1+n2-2))
    return (g1.mean() - g2.mean()) / pooled

fig2, ax = plt.subplots(figsize=(10, 5))
fig2.suptitle("Taille d'Effet (Cohen's d) - Hommes vs Femmes", fontsize=14, fontweight='bold')

d_lang = calc_cohens_d(df[df['subject']=='language']['male'], df[df['subject']=='language']['female'])
d_math = calc_cohens_d(df[df['subject']=='math']['male'], df[df['subject']=='math']['female'])

d_values = [('Language', d_lang), ('Math', d_math)]
y_pos = np.arange(len(d_values))

# Couleurs selon la taille d'effet
def get_color(d):
    if abs(d) < 0.2: return 'lightgreen'
    elif abs(d) < 0.5: return 'yellow'
    elif abs(d) < 0.8: return 'orange'
    else: return 'red'

bar_colors = [get_color(d) for _, d in d_values]
bars = ax.barh(y_pos, [d for _, d in d_values], color=bar_colors, edgecolor='black', height=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(['Language', 'Math'], fontsize=12)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
ax.axvline(x=0.2, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(x=-0.2, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.axvline(x=-0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.axvline(x=0.8, color='gray', linestyle='-.', alpha=0.5, linewidth=1)
ax.axvline(x=-0.8, color='gray', linestyle='-.', alpha=0.5, linewidth=1)

ax.set_xlabel("Cohen's d (positif = Hommes > Femmes)", fontsize=11)

# Ajouter les valeurs
for i, (name, d) in enumerate(d_values):
    color = 'black' if d > 0 else 'black'
    ax.text(d + 0.08, i, f'd = {d:.3f}', va='center', fontsize=13, fontweight='bold', color=color)

# Légende des seuils
ax.text(0.2, -0.8, '│', va='center', ha='center', fontsize=20, color='gray')
ax.text(0.2, -1, 'petit', va='top', ha='center', fontsize=10, color='gray')
ax.text(0.5, -0.8, '│', va='center', ha='center', fontsize=20, color='gray')
ax.text(0.5, -1, 'moyen', va='top', ha='center', fontsize=10, color='gray')
ax.text(0.8, -0.8, '│', va='center', ha='center', fontsize=20, color='gray')
ax.text(0.8, -1, 'grand', va='top', ha='center', fontsize=10, color='gray')
ax.set_ylim(-1.5, len(d_values))

plt.tight_layout()
plt.show()

In [27]:

print("\n" + "═"*80)
print("📏 STATISTIQUES NUMÉRIQUES PAR GENRE ET MATIÈRE")
print("═"*80)

summary_stats = df.groupby(['subject', 'gender'])['score'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('std', 'std'),
    ('median', 'median'),
    ('min', 'min'),
    ('max', 'max')
]).round(2)

print(summary_stats)


════════════════════════════════════════════════════════════════════════════════
📏 STATISTIQUES NUMÉRIQUES PAR GENRE ET MATIÈRE
════════════════════════════════════════════════════════════════════════════════
                 count   mean    std  median   min    max
subject  gender                                          
language female    245  73.00  14.01    73.0  29.5  100.0
         male      241  64.93  13.75    65.5  19.5  100.0
math     female    245  63.86  15.26    64.0  23.0  100.0
         male      241  68.95  14.43    70.0  30.0  100.0


In [ ]:
#les test d'hypothese pour comparer les moyennes entre les groupes
print('\n' + "═"*80)
print("📊 TESTS D'HYPOTHÈSE : hommes vs femmes")
print('\n' + "═"*80)

def cohens_d(group1, group2):
    """Calcule la taille d'effet (Cohen's d)"""
    n1, n2 = len(group1), len(group2)
    mean1, mean2 = group1.mean(), group2.mean()
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (mean1 - mean2) / pooled_std

def interpret_cohens_d(d):
    """Interprète la taille d'effet de Cohen"""
    d = abs(d)
    if d < 0.2:
        return "négligeable"
    elif d < 0.5:
        return "petit"
    elif d < 0.8:
        return "moyen"
    else:
        return "grand"

for subject in ['language', 'math']:
    print(f"\n {subject.upper()}")
    print("-"*40)

    # extraire les scores pour chaque genre
    male = df[(df['subject']==subject) & (df['gender']=='male')]['score']
    female = df[(df['subject']==subject) & (df['gender']=='female')]['score']
    
    # 📊 0. NORMALITY CHECK (Shapiro-Wilk)
    shapiro_female = stats.shapiro(female)
    shapiro_male = stats.shapiro(male)
    print(f"Test de normalité (Shapiro-Wilk) :")
    print(f"  Femmes: W = {shapiro_female.statistic:.4f}, p = {shapiro_female.pvalue:.4f}")
    print(f"  Hommes: W = {shapiro_male.statistic:.4f}, p = {shapiro_male.pvalue:.4f}")
    normal_female = shapiro_female.pvalue > 0.05
    normal_male = shapiro_male.pvalue > 0.05
    if normal_female and normal_male:
        print(f"  → Les deux groupes suivent une loi NORMALE ✅ (t-test valide)")
    else:
        print(f"  → Au moins un groupe N'EST PAS NORMAL ⚠️ (considérer Mann-Whitney U)")
    
    # 📊 1. T-TEST : comparaison des moyennes
    # Vérifier égalité des variances (Levene)
    levene_result = stats.levene(male, female)
    equal_var = levene_result.pvalue > 0.05  # Si p > 0.05 → variances égales
    
    # Lancer le t-test
    t_stat, p_ttest = stats.ttest_ind(male, female, equal_var=equal_var)
    
    # Calculer l'effect size (Cohen's d)
    d = cohens_d(male, female)
    d_interpretation = interpret_cohens_d(d)
    
    print(f"\nTest comparaison des moyennes (t-test) :")
    print(f"  moyenne_female : {female.mean():.2f} | moyenne_male : {male.mean():.2f}")
    print(f"  Difference : {female.mean() - male.mean():+.2f} points")
    print(f"  t = {t_stat:.3f} | p-value = {p_ttest:.4f}")
    print(f"  Cohen's d = {d:.3f} → effet {d_interpretation}")
    
    # Interpretation du t-test (comparaison des moyennes)
    if p_ttest < 0.05:
        print(f"  → DIFFERENCE SIGNIFICATIVE (p < 0.05)")
        if female.mean() > male.mean():
            print(f"    Les femmes ont un score moyen significativement supérieur")
        else:
            print(f"    Les hommes ont un score moyen significativement supérieur")
    else:
        print(f"  → PAS DE DIFFERENCE SIGNIFICATIVE (p >= 0.05)")
    
    # 📊 2. CHI-2 : test d'indépendance (distribution des catégories)
    chi = pd.crosstab(
        df[df['subject'] == subject]['score_cat'],
        df[df['subject'] == subject]['gender']
    )
    chi2_stat, p_chi2, dof, expected = stats.chi2_contingency(chi)
    print(f"\nTest d'indépendance (Chi-2) :")
    print(f"  Tableau observé :")
    print(chi)
    print(f"  χ² = {chi2_stat:.3f} | df = {dof} | p-value = {p_chi2:.4f}")
    if p_chi2 < 0.05:
        print(f"  → DISTRIBUTION DIFFÉRENTE entre genres (p < 0.05)")
    else:
        print(f"  → DISTRIBUTION SIMILAIRE entre genres (p >= 0.05)")

In [ ]:
# =============================================================================
# VISUALISATION DES TESTS D'HYPOTHÈSE
# =============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Tests Statistiques : Comparaison Hommes vs Femmes', fontsize=16, fontweight='bold')

subjects = ['language', 'math']
colors = {'female': '#FF69B4', 'male': '#4169E1'}

for idx, subject in enumerate(subjects):
    subset = df[df['subject'] == subject]
    female_data = subset[subset['gender'] == 'female']['score']
    male_data = subset[subset['gender'] == 'male']['score']
    
    # 1. BOXPLOT + MOYENNES
    ax1 = axes[idx, 0]
    sns.boxplot(data=subset, x='gender', y='score', ax=ax1, palette=colors, width=0.5)
    sns.stripplot(data=subset, x='gender', y='score', ax=ax1, color='black', alpha=0.3, size=4, jitter=True)
    
    # Ajouter moyennes
    female_mean = female_data.mean()
    male_mean = male_data.mean()
    ax1.scatter(0, female_mean, color='red', s=150, marker='★', label=f'Moyenne F: {female_mean:.1f}')
    ax1.scatter(1, male_mean, color='darkred', s=150, marker='★', label=f'Moyenne H: {male_mean:.1f}')
    ax1.legend(loc='upper right')
    ax1.set_title(f'{subject.title()} - Distribution des Scores', fontweight='bold', fontsize=12)
    ax1.set_ylabel('Score')
    ax1.set_xlabel('')
    
    # 2. BAR CHART - CATÉGORIES
    ax2 = axes[idx, 1]
    categories = ['Low (<50)', 'Medium (50-75)', 'High (>75)']
    crosstab_pct = pd.crosstab(subset['score_cat'], subset['gender'], normalize='columns') * 100
    
    x = np.arange(len(categories))
    width = 0.35
    
    female_pct = [crosstab_pct.loc[cat, 'female'] for cat in categories]
    male_pct = [crosstab_pct.loc[cat, 'male'] for cat in categories]
    
    bars1 = ax2.bar(x - width/2, female_pct, width, label='Femmes', color=colors['female'], edgecolor='black')
    bars2 = ax2.bar(x + width/2, male_pct, width, label='Hommes', color=colors['male'], edgecolor='black')
    
    ax2.set_ylabel('Pourcentage (%)')
    ax2.set_title(f'{subject.title()} - Répartition par Catégorie', fontweight='bold', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(categories, rotation=10, ha='right')
    ax2.legend()
    ax2.set_ylim(0, 100)
    
    # Valeurs sur les barres
    for bar in bars1:
        h = bar.get_height()
        ax2.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                    textcoords='offset points', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        h = bar.get_height()
        ax2.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                    textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# 3. FOREST PLOT - EFFECT SIZE (COHEN'S D)
def calc_cohens_d(g1, g2):
    n1, n2 = len(g1), len(g2)
    v1, v2 = g1.var(ddof=1), g2.var(ddof=1)
    pooled = np.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1+n2-2))
    return (g1.mean() - g2.mean()) / pooled

fig2, ax = plt.subplots(figsize=(10, 5))
fig2.suptitle("Taille d'Effet (Cohen's d) - Hommes vs Femmes", fontsize=14, fontweight='bold')

d_lang = calc_cohens_d(df[df['subject']=='language']['male'], df[df['subject']=='language']['female'])
d_math = calc_cohens_d(df[df['subject']=='math']['male'], df[df['subject']=='math']['female'])

d_values = [('Language', d_lang), ('Math', d_math)]
y_pos = np.arange(len(d_values))

# Couleurs selon la taille d'effet
def get_color(d):
    if abs(d) < 0.2: return 'lightgreen'
    elif abs(d) < 0.5: return 'yellow'
    elif abs(d) < 0.8: return 'orange'
    else: return 'red'

bar_colors = [get_color(d) for _, d in d_values]
bars = ax.barh(y_pos, [d for _, d in d_values], color=bar_colors, edgecolor='black', height=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(['Language', 'Math'], fontsize=12)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
ax.axvline(x=0.2, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(x=-0.2, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.axvline(x=-0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.axvline(x=0.8, color='gray', linestyle='-.', alpha=0.5, linewidth=1)
ax.axvline(x=-0.8, color='gray', linestyle='-.', alpha=0.5, linewidth=1)

ax.set_xlabel("Cohen's d (positif = Hommes > Femmes)", fontsize=11)

# Ajouter les valeurs
for i, (name, d) in enumerate(d_values):
    ax.text(d + 0.08, i, f'd = {d:.3f}', va='center', fontsize=13, fontweight='bold')

# Légende des seuils
ax.text(0.2, -0.8, '│', va='center', ha='center', fontsize=20, color='gray')
ax.text(0.2, -1, 'petit', va='top', ha='center', fontsize=10, color='gray')
ax.text(0.5, -0.8, '│', va='center', ha='center', fontsize=20, color='gray')
ax.text(0.5, -1, 'moyen', va='top', ha='center', fontsize=10, color='gray')
ax.text(0.8, -0.8, '│', va='center', ha='center', fontsize=20, color='gray')
ax.text(0.8, -1, 'grand', va='top', ha='center', fontsize=10, color='gray')
ax.set_ylim(-1.5, len(d_values))

plt.tight_layout()
plt.show()